# Validación Entorno Inicial

Antes de trabajar con datos se comprueba que el cuaderno se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías declaradas en `requirements.txt` están disponibles. La configuración del entorno se describe en la sección *Configurar el entorno* del `README.md`; aquí solo se verifica.

In [1]:
# PASO 4: Definir y ejecutar la verificación del entorno.
# Solo usa la biblioteca estándar: puede copiarse como primera celda de cualquier cuaderno.
import importlib
import platform
import sys
from pathlib import Path

# Librerías del proyecto (nombre de importación). scikit-learn se importa como sklearn.
LIBRERIAS_PROYECTO = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]


def verificar_entorno(librerias):
    '''
    Muestra y devuelve el estado del entorno de ejecución del cuaderno.

    Parámetros
    ----------
    librerias : list of str
        Nombres de importación de las librerías que el cuaderno necesita.

    Retorna
    -------
    dict
        Intérprete, si hay entorno virtual activo, carpeta de trabajo, versión de
        Python, sistema operativo y versión de cada librería (None si falta).
    '''
    # El intérprete que ejecuta ESTE cuaderno. Si no es el de .venv,
    # el kernel es el equivocado: Kernel -> Change Kernel.
    en_venv = sys.prefix != sys.base_prefix
    reporte = {
        "interprete": sys.executable,
        "entorno_virtual": en_venv,
        # Las rutas relativas se resuelven desde aquí, no desde la raíz del proyecto.
        "carpeta_trabajo": str(Path.cwd()),
        "python": platform.python_version(),
        "sistema": f"{platform.system()} {platform.release()}",
        "librerias": {},
    }

    print("Intérprete        :", reporte["interprete"])
    print("Entorno virtual   :", "[OK] activo" if en_venv else "[AVISO] se usa el Python del sistema")
    print("Carpeta de trabajo:", reporte["carpeta_trabajo"])
    print("Python            :", reporte["python"])
    print("Sistema operativo :", reporte["sistema"])

    print("\nLibrerías")
    for nombre in librerias:
        try:
            version = importlib.import_module(nombre).__version__
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            version = None
            print(f"  [FALTA] {nombre:12} instalar con: python -m pip install -r requirements.txt")
        reporte["librerias"][nombre] = version

    return reporte


ENTORNO = verificar_entorno(LIBRERIAS_PROYECTO)

Intérprete        : /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500/.venv/bin/python
Entorno virtual   : [OK] activo
Carpeta de trabajo: /Users/oscar/Documents/Magister CS e IA/repositorio/proyecto-grupo2-mcdi500/F2
Python            : 3.12.3
Sistema operativo : Darwin 25.5.0

Librerías
  [OK]    numpy        2.5.4
  [OK]    pandas       3.1.0
  [OK]    matplotlib   3.11.2
  [OK]    seaborn      0.13.2
  [OK]    sklearn      1.9.1


# Fase 2: exploración y preprocesamiento

Este notebook carga el archivo original, revisa su calidad y prepara una copia de trabajo con fechas y montos convertidos. El archivo CSV original se conserva sin modificaciones.

**Recorrido:** `CSV original` → `revisión de estructura` → `inspección de valores` → `conversiones controladas` → `resúmenes para validar`

En cada bloque, las tablas y conteos permiten comprobar qué se leyó o transformó antes de continuar.

## 1. Cargar el archivo y comprobar la lectura

`CSV` → `DataFrame df` → `dimensiones y muestra`

Se importa `pandas` y se busca el archivo en `data/raw/`, empezando por el directorio de trabajo y sus carpetas superiores. Así, la ruta funciona tanto al ejecutar desde `F2/` como desde la raíz. Luego se lee el CSV detectando su separador y usando la codificación indicada. La salida muestra las dimensiones y las primeras cinco filas para revisar encabezados y caracteres.

In [2]:
# PASO 1: Importar las herramientas necesarias.
# Path permite trabajar con rutas de carpetas y archivos.
from pathlib import Path

# pandas permite cargar y analizar datos en tablas.
import pandas as pd


# PASO 2: Buscar el archivo de datos desde el directorio de trabajo.
# Esto permite ejecutar el notebook tanto desde su carpeta como desde la raíz.
nombre_archivo = "04LicitacionesEne.csv"
ruta = Path.cwd() / "data" / "raw" / nombre_archivo

if not ruta.is_file():
    for directorio in Path.cwd().parents:
        candidata = directorio / "data" / "raw" / nombre_archivo
        if candidata.is_file():
            ruta = candidata
            break

if not ruta.is_file():
    raise FileNotFoundError(
        f"No se encontró {nombre_archivo} en data/raw desde {Path.cwd()}"
    )


# PASO 3: Cargar el CSV en una tabla llamada df.
# sep=None intenta detectar el separador del archivo.
# engine="python" permite esa detección automática.
# encoding="latin-1" es la codificación con la que lograste abrirlo.
# Este comando lee el archivo; no modifica el CSV original.
df = pd.read_csv(
    ruta,
    sep=None,
    engine="python",
    encoding="latin-1"
)


# PASO 4: Mostrar la cantidad de filas y columnas.
# shape[0] corresponde a las filas.
# shape[1] corresponde a las columnas.
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])


# PASO 5: Mostrar las primeras cinco filas.
# Revisaremos que las columnas y las tildes se lean correctamente.
display(df.head())

Cantidad de filas: 45741
Cantidad de columnas: 74


,NroLicitacion,NombreLicitacion,TipoLicitacion,Descripcion,MonedaLicitacion,MontoEstimadoLicitacion,MontoEstimadoVisible,BaseEstimacionMontoLicitacion,FuenteFinanciamiento,JustificacionMontoEstimado,...,ActividadProveedor,TamanoProveedor,NombreOferta,EspecificacionesProveedor,EstadoOferta,CantidadOferta,MonedaOferta,MontoNetoOferta,MontoTotalOferta,ResultadoOferta
0,2366-129-L124,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,Licitación Pública Menor a 100 UTM (L1),MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,CLP,3630470,Si,Precio Referencial,NaN,1,...,VENTA AL POR MENOR DE OTROS PRODUCTOS EN COMER...,Mediana,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,"Cajonera con llave alto 70, ancho 50, prof. 5...",Aceptada,8,CLP,115879,927032,Perdedora
1,2366-129-L124,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,Licitación Pública Menor a 100 UTM (L1),MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,CLP,3630470,Si,Precio Referencial,NaN,1,...,FABRICACION DE MUEBLES PRINCIPALMENTE DE MADERA,NoClasificado,MUEBLES,MUEBLE DE 2 PUERTAS,Aceptada,5,CLP,150000,750000,Perdedora
2,2366-129-L124,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,Licitación Pública Menor a 100 UTM (L1),MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,CLP,3630470,Si,Precio Referencial,NaN,1,...,VENTA AL POR MENOR DE OTROS PRODUCTOS EN COMER...,Mediana,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,Estante sin puertas 5 divisiones color blanco....,Aceptada,1,CLP,465931,465931,Perdedora
3,2366-129-L124,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,Licitación Pública Menor a 100 UTM (L1),MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,CLP,3630470,Si,Precio Referencial,NaN,1,...,FABRICACION DE MUEBLES PRINCIPALMENTE DE MADERA,NoClasificado,MUEBLES,MUEBLE TIPO BODEGA 2,Aceptada,1,CLP,270000,270000,Perdedora
4,2366-129-L124,MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,Licitación Pública Menor a 100 UTM (L1),MUEBLES VARIOS CESFAM ROSA VASCOPE SC N°854 Y ...,CLP,3630470,Si,Precio Referencial,NaN,1,...,"Servicios de impresión, consultoría y capacita...",Pequeña,PROPUESTA DE SUMINISTRO MUEBLES VARIOS,Kardex 4 Cajones Carpetas Colgantes Oficina Bl...,Aceptada,3,CLP,295074,885222,Ganadora


## 2. Perfilar columnas y detectar duplicados

`df` → `tipos + nulos + valores distintos + duplicados`

La tabla resume cada columna con su tipo, cantidad y porcentaje de nulos y número de valores únicos. Al final se cuenta cuántas filas son idénticas; solo se reportan, no se eliminan.

In [3]:
# PASO 6: Crear un resumen de las columnas de la base.
resumen = pd.DataFrame({
    
    # Mostrar el tipo que pandas reconoció en cada columna.
    "Tipo de dato": df.dtypes.astype(str),
    
    # Contar los valores faltantes de cada columna.
    "Valores nulos": df.isna().sum(),
    
    # Calcular el porcentaje de valores faltantes.
    "Porcentaje nulos": (df.isna().mean() * 100).round(2),
    
    # Contar cuántos valores diferentes tiene cada columna.
    "Valores distintos": df.nunique()
})

# PASO 7: Mostrar hasta 10 filas del resumen para no saturar la salida.
display(resumen.head(10))


# PASO 8: Contar las filas que son completamente iguales.
# Todavía no las eliminamos: primero debemos revisar su significado.
duplicados = df.duplicated().sum()

print("Filas completamente duplicadas:", duplicados)

,Tipo de dato,Valores nulos,Porcentaje nulos,Valores distintos
NroLicitacion,str,0,0.00,2292
NombreLicitacion,str,0,0.00,2282
TipoLicitacion,str,0,0.00,9
Descripcion,str,0,0.00,2222
MonedaLicitacion,str,0,0.00,4
MontoEstimadoLicitacion,int64,0,0.00,1786
MontoEstimadoVisible,str,0,0.00,2
BaseEstimacionMontoLicitacion,str,0,0.00,3
FuenteFinanciamiento,str,20012,43.75,609
JustificacionMontoEstimado,str,29854,65.27,495


Filas completamente duplicadas: 0


## 3. Inspeccionar fechas, montos y filas de una licitación

`columnas originales` → `muestras concretas` → `contexto por licitación`

Se muestran ejemplos de las tres fechas y de moneda y montos tal como vienen en el CSV. Luego se toma la primera licitación encontrada y se presentan hasta 15 filas con producto, proveedor y resultado para entender qué representa cada registro.

In [4]:
# PASO 9: Ver ejemplos de las fechas.
# Primero observamos su formato para convertirlas correctamente después.
columnas_fecha = [
    "FechaPublicacion",
    "FechaCierre",
    "FechaAdjudicacion"
]

display(df[columnas_fecha].head(10))


# PASO 10: Revisar cómo están escritos los montos y sus monedas.
# Todavía no reemplazamos puntos ni comas.
columnas_monto = [
    "MonedaOferta",
    "MontoNetoOferta",
    "MontoTotalOferta"
]

display(df[columnas_monto].head(10))


# PASO 11: Examinar varias filas de una misma licitación.
# Esto ayuda a entender qué representa cada fila de la base.
licitacion_ejemplo = df["NroLicitacion"].iloc[0]

columnas_detalle = [
    "NroLicitacion",
    "CodigoProductoONU",
    "NombreItem",
    "ProveedorRUT",
    "EstadoOferta",
    "ResultadoOferta"
]

display(
    df.loc[
        df["NroLicitacion"] == licitacion_ejemplo,
        columnas_detalle
    ].head(10)
)

,FechaPublicacion,FechaCierre,FechaAdjudicacion
0,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
1,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
2,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
3,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
4,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
5,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
6,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
7,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
8,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58
9,27-12-2024 11:07,07-01-2025 17:30,04-04-2025 14:58


,MonedaOferta,MontoNetoOferta,MontoTotalOferta
0,CLP,115879,927032
1,CLP,150000,750000
2,CLP,465931,465931
3,CLP,270000,270000
4,CLP,295074,885222
5,CLP,265000,265000
6,CLP,199900,199900
7,CLP,161308,322616
8,CLP,79900,159800
9,CLP,139900,139900


,NroLicitacion,CodigoProductoONU,NombreItem,ProveedorRUT,EstadoOferta,ResultadoOferta
0,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Perdedora
1,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Perdedora
2,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Perdedora
3,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Perdedora
4,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Ganadora
5,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Perdedora
6,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Ganadora
7,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Perdedora
8,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Ganadora
9,2366-129-L124,56111502,Muebles no modulares de oficina ejecutiva,[RUT],Aceptada,Ganadora


## 4. Convertir fechas en una copia de trabajo

`df original` → `df_trabajo` → `fechas datetime + control de fallos`

Se conserva `df` y se crea `df_trabajo` para transformar las fechas sin modificar los datos cargados. La tabla de control separa los nulos que ya existían de los valores presentes que no coinciden con el formato esperado; esos fallos quedan como `NaT`.

In [5]:
# PASO 12: Crear una copia para trabajar.
# df conserva los datos tal como se cargaron.
df_trabajo = df.copy()


# PASO 13: Indicar las columnas de fecha que revisamos.
columnas_fecha = [
    "FechaPublicacion",
    "FechaCierre",
    "FechaAdjudicacion"
]


# PASO 14: Convertir el texto a fechas.
# %d: día; %m: mes; %Y: año; %H: hora; %M: minutos.
# errors="coerce" convierte los valores no interpretables en NaT.
# Después contamos esos casos para no perderlos de vista.
revision_fechas = []

for columna in columnas_fecha:
    original = df[columna]

    df_trabajo[columna] = pd.to_datetime(
        original,
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

    # Detectar valores que existían, pero no pudieron convertirse.
    fallos = original.notna() & df_trabajo[columna].isna()

    revision_fechas.append({
        "Columna": columna,
        "Nulos originales": original.isna().sum(),
        "Fallos de conversión": fallos.sum()
    })


# PASO 15: Mostrar el control de la conversión.
display(pd.DataFrame(revision_fechas))

# Mostrar ejemplos de las fechas convertidas.
display(df_trabajo[columnas_fecha].head())

,Columna,Nulos originales,Fallos de conversión
0,FechaPublicacion,0,0
1,FechaCierre,0,0
2,FechaAdjudicacion,91,0


,FechaPublicacion,FechaCierre,FechaAdjudicacion
0,2024-12-27 11:07:00,2025-01-07 17:30:00,2025-04-04 14:58:00
1,2024-12-27 11:07:00,2025-01-07 17:30:00,2025-04-04 14:58:00
2,2024-12-27 11:07:00,2025-01-07 17:30:00,2025-04-04 14:58:00
3,2024-12-27 11:07:00,2025-01-07 17:30:00,2025-04-04 14:58:00
4,2024-12-27 11:07:00,2025-01-07 17:30:00,2025-04-04 14:58:00


## 5. Revisar monedas y formatos de montos

`MonedaOferta` → `frecuencias`  |  `montos` → `conteo y ejemplos especiales`

Primero se cuentan las monedas, incluidos los valores nulos. Después se inspeccionan los montos como texto y se listan ejemplos que contienen algo distinto de dígitos, como separadores o espacios. Esta revisión es diagnóstica: todavía no cambia los datos.

In [6]:
# PASO 16: Revisar las monedas presentes.
# Los montos de monedas diferentes deben analizarse por separado.
display(df_trabajo["MonedaOferta"].value_counts(dropna=False))


# PASO 17: Buscar montos con puntos, comas u otros caracteres.
# Esta revisión no modifica los valores.
for columna in ["MontoNetoOferta", "MontoTotalOferta"]:

    # Convertir temporalmente a texto y quitar espacios exteriores.
    texto = df_trabajo[columna].astype("string").str.strip()

    # Identificar valores que no son únicamente dígitos.
    tiene_formato_especial = ~texto.str.fullmatch(
        r"\d+", na=False
    ) & texto.notna()

    print("\nColumna:", columna)
    print("Valores con formato especial:", tiene_formato_especial.sum())

    # Mostrar hasta 10 ejemplos distintos para revisar el formato.
    display(
        texto[tiene_formato_especial]
        .drop_duplicates()
        .head(10)
        .to_frame(name="Ejemplos")
    )

MonedaOferta
CLP    45604
CLF      105
USD       23
UTM        9
Name: count, dtype: int64


Columna: MontoNetoOferta
Valores con formato especial: 1123


,Ejemplos
139,"10504,202"
191,"9,75"
241,"48,47"
415,"197,8"
453,"349,8"
481,"362,6"
516,"1002,7"
691,"32773109,24"
981,"29,8"
986,"47,9"



Columna: MontoTotalOferta
Valores con formato especial: 193


,Ejemplos
691,"32773109,24"
1090,"13663865,55"
1139,"355,85"
1324,"4939999,962"
1371,"1260,58"
1765,"714,05"
1854,"556,8"
2064,"318,72"
2103,"743,16"
3175,"6170579,832"


## 6. Convertir montos a valores numéricos

`texto con coma decimal` → `número` → `reporte de fallos y tipos`

En `df_trabajo`, la coma se reemplaza por punto y se intenta convertir cada monto a número. Los valores que no se puedan interpretar se convierten en nulos y se reportan, distinguiéndolos de los nulos originales. Los puntos existentes se conservan; por eso conviene revisar los formatos especiales identificados en el bloque anterior.

In [7]:
# PASO 18: Convertir los montos de texto a número.
# Trabajamos en df_trabajo; df conserva los valores originales.
revision_montos = []

for columna in ["MontoNetoOferta", "MontoTotalOferta"]:

    # Conservar el texto para controlar la conversión.
    original = df_trabajo[columna].astype("string").str.strip()

    # Cambiar la coma decimal por un punto.
    # No eliminamos puntos ni redondeamos los montos.
    texto_convertido = original.str.replace(",", ".", regex=False)

    # Convertir a número.
    # Los valores no interpretables quedan como nulos y se reportan.
    numeros = pd.to_numeric(texto_convertido, errors="coerce")

    # Detectar valores presentes que no pudieron convertirse.
    fallos = original.notna() & numeros.isna()

    revision_montos.append({
        "Columna": columna,
        "Nulos originales": original.isna().sum(),
        "Fallos de conversión": fallos.sum()
    })

    # Mostrar ejemplos si se detectan problemas.
    if fallos.any():
        print("Revisar estos valores de:", columna)
        display(original[fallos].drop_duplicates().head(20))

    # Guardar el resultado numérico en la copia de trabajo.
    df_trabajo[columna] = numeros


# PASO 19: Mostrar el control de la conversión.
display(pd.DataFrame(revision_montos))

# Comprobar los tipos de datos resultantes.
display(
    df_trabajo[["MontoNetoOferta", "MontoTotalOferta"]].dtypes
)

,Columna,Nulos originales,Fallos de conversión
0,MontoNetoOferta,0,0
1,MontoTotalOferta,0,0


MontoNetoOferta     Float64
MontoTotalOferta    Float64
dtype: object

## 7. Validar montos y revisar ofertas

`monto total` → `estadísticos por moneda` → `ceros/negativos` → `mayores ofertas CLP`

Se resumen cantidad, mínimo, mediana, promedio y máximo por moneda, sin mezclar unidades. Después se cuentan montos cero y negativos y se muestran las cinco ofertas más altas en CLP con sus datos de contexto. Son señales para revisar, no reglas automáticas para borrar registros.

In [8]:
# PASO 20: Resumir los montos por moneda.
# No mezclamos CLP, CLF, USD y UTM.
resumen_montos = (
    df_trabajo.groupby("MonedaOferta")["MontoTotalOferta"]
    .agg(
        Cantidad="count",
        Minimo="min",
        Mediana="median",
        Promedio="mean",
        Maximo="max"
    )
)

display(resumen_montos)


# PASO 21: Contar montos iguales a cero y negativos.
# Solo los identificamos; todavía no eliminamos ninguna fila.
for columna in ["MontoNetoOferta", "MontoTotalOferta"]:
    print("\nColumna:", columna)
    print("Montos iguales a cero:", (df_trabajo[columna] == 0).sum())
    print("Montos negativos:", (df_trabajo[columna] < 0).sum())


# PASO 22: Revisar las cinco ofertas de mayor monto en CLP.
# Un monto alto no es necesariamente un error.
ofertas_clp = df_trabajo[
    df_trabajo["MonedaOferta"] == "CLP"
]

display(
    ofertas_clp.nlargest(5, "MontoTotalOferta")[
        [
            "NroLicitacion",
            "NombreItem",
            "CantidadOferta",
            "MontoNetoOferta",
            "MontoTotalOferta",
            "ResultadoOferta"
        ]
    ]
)

,Cantidad,Minimo,Mediana,Promedio,Maximo
MonedaOferta,,,,,
CLF,105,1.0,449.16,104762480262.242966,10000000000000.0
CLP,45604,-1.0,259200.0,9211294.125976,61725500000.0
USD,23,0.176,30906.0,52443.723739,474798.0
UTM,9,2.8924,556.8,21708.926933,48617.65



Columna: MontoNetoOferta
Montos iguales a cero: 0
Montos negativos: 1

Columna: MontoTotalOferta
Montos iguales a cero: 0
Montos negativos: 1


,NroLicitacion,NombreItem,CantidadOferta,MontoNetoOferta,MontoTotalOferta,ResultadoOferta
20577,4053-1-LE25,Unidades dentales,1025,60220000.0,61725500000.0,Perdedora
42052,2424-4-LE25,Estuches o bolsas de primeros auxilios para lo...,850,9103500.0,7737975000.0,Perdedora
949,3706-104-LR24,Construcción de obras civiles,1,3950223199.0,3950223199.0,Perdedora
23209,3706-104-LR24,Construcción de obras civiles,1,3101220959.0,3101220959.0,Ganadora
391,2319-20-LR24,Construcción de obras civiles,1,2727024140.0,2727024140.0,Perdedora


## 8. Revisar categorías y nombres equivalentes

`variables categóricas` → `frecuencias` | `texto normalizado` → `comparación de cardinalidad` → `RUT vs nombre`

Se inspeccionan las categorías principales, se compara la cantidad de valores distintos antes y después de normalizar mayúsculas/espacios y se verifica si un mismo RUT de proveedor registra múltiples nombres.

In [9]:
# PASO 23: Revisar los valores únicos de las variables categóricas clave.
# Esto permite ver cómo están escritas las categorías de TamanoProveedor y otras columnas.
columnas_categoricas = [
    "TamanoProveedor",
    "ResultadoOferta",
    "EstadoOferta",
    "EstadoLicitacion",
    "TipoLicitacion"
]

for columna in columnas_categoricas:
    print(f"\n--- Valores en {columna} ---")
    display(df_trabajo[columna].value_counts(dropna=False).to_frame(name="Frecuencia"))


# PASO 24: Buscar nombres equivalentes escritos con distintas mayúsculas o espacios.
# Comparamos cuántos valores distintos hay antes y después de normalizar el texto.
columnas_nombres = [
    "TamanoProveedor",
    "Proveedor",
    "Institucion",
    "UnidadCompra",
    "UnidadMedida",
    "NombreItem"
]

revision_equivalentes = []

for columna in columnas_nombres:
    original = df_trabajo[columna].astype("string")
    normalizado = original.str.strip().str.upper()

    distintos_original = original.nunique(dropna=True)
    distintos_normalizado = normalizado.nunique(dropna=True)

    revision_equivalentes.append({
        "Columna": columna,
        "Distintos originales": distintos_original,
        "Distintos normalizados": distintos_normalizado,
        "Diferencia": distintos_original - distintos_normalizado
    })

display(pd.DataFrame(revision_equivalentes))


# PASO 25: Revisar si un mismo RUT de proveedor tiene más de un nombre registrado.
nombres_por_rut = (
    df_trabajo.groupby("ProveedorRUT")["Proveedor"]
    .nunique()
)

ruts_con_varios_nombres = nombres_por_rut[nombres_por_rut > 1]
print("RUTs de proveedor con más de un nombre:", len(ruts_con_varios_nombres))


--- Valores en TamanoProveedor ---


,Frecuencia
TamanoProveedor,
Pequeña,13510
Grande,11848
Mediana,8570
Micro,6539
NoClasificado,5274



--- Valores en ResultadoOferta ---


,Frecuencia
ResultadoOferta,
Perdedora,25853
Ganadora,19888



--- Valores en EstadoOferta ---


,Frecuencia
EstadoOferta,
Aceptada,41860
Rechazada,3881


... (2 outputs left out)


--- Valores en TipoLicitacion ---


,Frecuencia
TipoLicitacion,
Licitación Pública Entre 100 y 1000 UTM (LE),32569
Licitación Pública Menor a 100 UTM (L1),7594
Licitación Pública entre a 2000 y 5000 UTM (LQ),2026
Licitación Pública Mayor a 5000 (LR),1836
Licitación Pública Mayor 1000 UTM (LP),1664
Licitación Privada entre 100 y 1000 UTM.,31
Licitación Privada Menor a 100 UTM.,10
Licitación Privada Mayor a 1000 UTM,6
Licitación Pública Servicios personales especializados (LS),5


,Columna,Distintos originales,Distintos normalizados,Diferencia
0,TamanoProveedor,5,5,0
1,Proveedor,4700,4699,1
2,Institucion,326,326,0
3,UnidadCompra,419,368,51
4,UnidadMedida,47,47,0
5,NombreItem,1697,1697,0


RUTs de proveedor con más de un nombre: 0


## 9. Confirmar el orden de TamanoProveedor y muestra aleatoria

`TamanoProveedor` → `CategoricalDtype ordenado` → `verificación de conteos` → `muestra aleatoria de 10 filas`

Se establece el orden lógico de tamaño de proveedor (`Micro`, `Pequeña`, `Mediana`, `Grande`, `NoClasificado`) y se extrae una muestra aleatoria de 10 registros ordenada por esta variable.

In [10]:
# PASO 26: Definir el orden lógico de TamanoProveedor.
# En el Paso 23 vimos que la categoría viene escrita como "Micro".
orden_tamano = [
    "Micro",
    "Pequeña",
    "Mediana",
    "Grande",
    "NoClasificado"
]

# Verificar que ningún valor de la columna quede fuera de la lista definida.
valores_actuales = set(df_trabajo["TamanoProveedor"].dropna().unique())
valores_no_incluidos = valores_actuales - set(orden_tamano)
print("Valores no incluidos en el orden:", valores_no_incluidos)


# PASO 27: Convertir TamanoProveedor a categoría ordenada y confirmar el orden.
if not valores_no_incluidos:
    tipo_tamano = pd.CategoricalDtype(categories=orden_tamano, ordered=True)
    df_trabajo["TamanoProveedor"] = df_trabajo["TamanoProveedor"].astype(tipo_tamano)

    # Mostrar el conteo respetando el nuevo orden establecido (sort_index).
    display(
        df_trabajo["TamanoProveedor"]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame(name="Cantidad")
    )


# PASO 28: Generar una muestra provisional de 10 ejemplos aleatorios de la base.
# Usamos un generador aleatorio (default_rng) con semilla fija para que la muestra sea reproducible.
import numpy as np

columnas_ejemplo = [
    "NroLicitacion",
    "Proveedor",
    "ProveedorRUT",
    "TamanoProveedor",
    "MontoTotalOferta",
    "ResultadoOferta"
]

SEMILLA = 42  # semilla fija: la muestra es la misma en cada ejecución
rng = np.random.default_rng(SEMILLA)

muestra_provisional = (
    df_trabajo[columnas_ejemplo]
    .sample(n=10, random_state=rng)
    .sort_values("TamanoProveedor")
)

print("\nMuestra provisional de 10 ejemplos aleatorios (ordenados por TamanoProveedor):")
display(muestra_provisional)

Valores no incluidos en el orden: set()


,Cantidad
TamanoProveedor,
Micro,6539
Pequeña,13510
Mediana,8570
Grande,11848
NoClasificado,5274



Muestra provisional de 10 ejemplos aleatorios (ordenados por TamanoProveedor):


,NroLicitacion,Proveedor,ProveedorRUT,TamanoProveedor,MontoTotalOferta,ResultadoOferta
39269,3987-1-LE25,GOAT PHARMA SPA,[RUT],Micro,410000.0,Ganadora
3930,2311-2-L125,SOCIEDAD COMERCIAL FARMACÉUTICA BRAVO CABRERA ...,[RUT],Pequeña,770000.0,Perdedora
35395,3558-2-LE25,"SOCIEDAD COMERCIAL, SERVICIOS E INVERSIONES AU...",[RUT],Pequeña,30000.0,Ganadora
29936,514862-180-LE24,EXPRO SPA,[RUT],Mediana,323100.0,Perdedora
19804,2324-7-LQ25,LOJAN SPA,[RUT],Mediana,259440000.0,Ganadora
31896,3692-1-LE25,"INGENIERIA, SOPORTE Y CADENA DE SUMINISTRO SPA",[RUT],Mediana,216000.0,Ganadora
4081,3251-33-LQ24,NIPRO MEDICAL CORPORATION (AGENCIA EN CHILE),[RUT],Grande,37500.0,Ganadora
20072,3492-1-LE25,ITF-LABOMED FARMACEUTICA LIMITADA,[RUT],Grande,90000.0,Perdedora
4307,2387-225-L124,SOC MUEBLES SANTA ANA LIMITADA,[RUT],Grande,97000.0,Perdedora
9215,2395-5-L125,MUNNICH PHARMA MEDICAL LIMITADA,[RUT],Grande,33600.0,Perdedora


## 10. Detectar valores atípicos mediante rango intercuartílico (IQR)

`ofertas CLP` → `Q1, Q3 e IQR` → `límites 1.5 * IQR` → `muestra de 10 valores atípicos`

Se calcula el rango intercuartílico para `MontoTotalOferta` en CLP sin mezclar monedas, se contabilizan las ofertas fuera de los límites y se despliega una muestra aleatoria de 10 casos atípicos para su revisión.

In [11]:
# PASO 29: Calcular el rango intercuartílico (IQR) para MontoTotalOferta en CLP.
# Analizamos CLP por separado para no mezclar distintas monedas.
ofertas_clp = df_trabajo[df_trabajo["MonedaOferta"] == "CLP"].copy()

q1 = ofertas_clp["MontoTotalOferta"].quantile(0.25)
q3 = ofertas_clp["MontoTotalOferta"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - (1.5 * iqr)
limite_superior = q3 + (1.5 * iqr)

resumen_iqr = pd.DataFrame([{
    "Moneda": "CLP",
    "Q1 (25%)": q1,
    "Q3 (75%)": q3,
    "IQR": iqr,
    "Límite inferior": limite_inferior,
    "Límite superior": limite_superior
}])

display(resumen_iqr)


# PASO 30: Identificar los valores atípicos según el criterio 1.5 * IQR.
# Solo los detectamos para analizarlos; no los eliminamos automáticamente.
es_atipico = (
    (ofertas_clp["MontoTotalOferta"] < limite_inferior)
    | (ofertas_clp["MontoTotalOferta"] > limite_superior)
)

atipicos_clp = ofertas_clp[es_atipico]
porcentaje_atipicos = (len(atipicos_clp) / len(ofertas_clp)) * 100

print(f"Total de ofertas en CLP: {len(ofertas_clp)}")
print(f"Valores atípicos detectados: {len(atipicos_clp)} ({porcentaje_atipicos:.2f}%)")


# PASO 31: Mostrar un ejemplo de 10 valores atípicos obtenidos.
import numpy as np

columnas_atipicos = [
    "NroLicitacion",
    "NombreItem",
    "Proveedor",
    "TamanoProveedor",
    "CantidadOferta",
    "MontoNetoOferta",
    "MontoTotalOferta",
    "ResultadoOferta"
]

rng = np.random.default_rng(SEMILLA)  # misma semilla del Paso 28
muestra_atipicos = atipicos_clp[columnas_atipicos].sample(n=10, random_state=rng)

print("\nMuestra de 10 valores atípicos detectados en CLP:")
display(muestra_atipicos)

,Moneda,Q1 (25%),Q3 (75%),IQR,Límite inferior,Límite superior
0,CLP,43600.0,1801754.5,1758154.5,-2593631.75,4438986.25


Total de ofertas en CLP: 45604
Valores atípicos detectados: 7636 (16.74%)

Muestra de 10 valores atípicos detectados en CLP:


,NroLicitacion,NombreItem,Proveedor,TamanoProveedor,CantidadOferta,MontoNetoOferta,MontoTotalOferta,ResultadoOferta
3034,4086-114-LE24,Construcción de obras civiles,PREVENGO SPA,Micro,1,7000000.0,7000000.0,Perdedora
35373,2428-3-LP25,Artículos de papelería,DIMERC S A,Grande,1000,9422.0,9422000.0,Perdedora
3149,2345-520-LE24,"Notebook, laptop o computador portátil excepto...",IMPORTADORA Y COMERCIALIZADORA CHIVENSU SPA,NoClasificado,3,2500575.0,7501725.0,Perdedora
29596,816696-1-LE25,Profesionales,[PERSONA],NoClasificado,1,9980000.0,9980000.0,Perdedora
19760,2698-1-LE25,Servicio de mantenimiento de equipo de fabrica...,PETRINOVIC SPA,Mediana,1,15472800.0,15472800.0,Ganadora
19386,3511-1-LP25,Libros o cuadernos de registro,DISTRIBUIDORA SEGUEL LTDA,Pequeña,1,69581214.0,69581214.0,Perdedora
31643,3287-109-LE24,Generadores diesel,SEINCO SERVICIOS DE INVERSIONES INGENIERÍA Y C...,NoClasificado,1,20400000.0,20400000.0,Perdedora
3369,514862-183-LE24,Unidades dentales,DENTAL LAVAL LIMITADA,Grande,150,138000.0,20700000.0,Perdedora
9095,3759-2-LE25,Caja,[PERSONA],Pequeña,385,28025.0,10789625.0,Ganadora
39214,2719-9-LE25,Mobiliario,INGENIERIA CONSTRUCCION Y SERVICIOS BALOC SPA,Micro,1,19916040.0,19916040.0,Ganadora


## 11. Guardar base preprocesada en data/processed

`df_trabajo` → `data/processed/04LicitacionesEne_procesado.csv` → `verificación de exportación`

Se guarda la tabla `df_trabajo` con las fechas, montos y categorías ya procesadas dentro de la carpeta compartida `data/processed/`, manteniendo el archivo original intacto en `data/raw/`.

In [12]:
# PASO 32: Definir la ruta de salida en la carpeta compartida data/processed.
# Usamos la ubicación de ruta (data/raw) para encontrar data/processed sin importar desde dónde se ejecute.
carpeta_procesados = ruta.parent.parent / "processed"
carpeta_procesados.mkdir(parents=True, exist_ok=True)

ruta_salida = carpeta_procesados / "04LicitacionesEne_procesado.csv"


# PASO 33: Exportar df_trabajo a CSV y comprobar que el archivo se creó correctamente.
# Guardamos con separador ';' y codificación 'utf-8-sig' para conservar tildes y ñ.
df_trabajo.to_csv(
    ruta_salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

tamano_mb = ruta_salida.stat().st_size / (1024 * 1024)
print("Archivo guardado correctamente en:", ruta_salida.relative_to(ruta.parents[2]).as_posix())
print(f"Dimensiones exportadas: {df_trabajo.shape[0]} filas y {df_trabajo.shape[1]} columnas")
print(f"Tamaño del archivo: {tamano_mb:.2f} MB")

Archivo guardado correctamente en: data/processed/04LicitacionesEne_procesado.csv
Dimensiones exportadas: 45741 filas y 74 columnas
Tamaño del archivo: 59.80 MB
